In [1]:
import torch.nn.functional  as F
import torch.nn as nn
import torch

### MHLA (Multi-Head Latent Attention)

`MultiHeadLatentAttention` module, here are the key aspects:

**Core Innovation**: This compresses the attention mechanism by projecting inputs to a lower-dimensional latent space before computing attention, reducing computational cost compared to standard multi-head attention.

**Two-Stage Projection**:
- **Down projections**: Compress Q, K, V from `hidden_size` to `latent_dim` (4x smaller)
- **Up projections**: Expand back for attention computation (K and Q to `latent_dim//2`, V to full `hidden_size`)

**Dual-Stream Keys & Queries**: Each has two components concatenated together:
- A **context part** (from the up-projections)
- A **RoPE part** (rotary position embeddings applied separately)

This split allows positional information to be added without affecting the compressed representations.

**Standard Multi-Head Structure**: After projections, it splits into multiple heads, applies scaled dot-product attention with causal masking, then projects back through the output layer.

In [2]:
class MultiHeadLatentAttention(nn.Module):

    def __init__(self,hidden_size,num_heads,compression_ratio=4):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.latent_dim = hidden_size//compression_ratio
        self.head_dim = hidden_size//num_heads

        # down projections
        self.kv_proj_d = nn.Linear(self.hidden_size,self.latent_dim, bias=False)
        self.q_proj_d = nn.Linear(self.hidden_size,self.latent_dim, bias=False)

        # up projections
        self.k_proj_u = nn.Linear(self.latent_dim,self.latent_dim//2, bias=False)
        self.q_proj_u = nn.Linear(self.latent_dim,self.latent_dim//2, bias=False)
        self.v_proj_u = nn.Linear(self.latent_dim,self.hidden_size, bias=False)

        # out projection
        self.o_proj = nn.Linear(self.hidden_size,self.hidden_size, bias=False)

        # rope_k and rope_q
        self.rope_k =  nn.Linear(self.hidden_size, self.hidden_size//2, bias=False)
        self.rope_q =  nn.Linear(self.latent_dim, self.hidden_size//2, bias=False)

        self.rotary_emb = LlamaRotaryEmbedding(self.head_dim//2)

    def forward(self,x,attention_mask=None):# x is [batch_size, seq_len,hidden_size]

        # Extract the batch_size, seq_len as we will need them later for tensor operations
        batch_size,seq_len,_ = x.shape

        # down project kv
        kv_d = self.kv_proj_d(x)
        # down project q
        q_d = self.q_proj_d(x)

        # Applying the up projections
        k_proj_2 = self.k_proj_u(kv_d)
        v_proj_2 = self.v_proj_u(kv_d)
        q_proj_2 = self.q_proj_u(q_d)

        # Apply RoPEprojections
        k_rope_2 = self.rope_k(x)
        q_rope_2 = self.rope_q(q_d)

        # Creating Multiple Heads
        k_proj_2 = k_proj_2.view(batch_size,seq_len,self.num_heads,-1)
        q_proj_2 = q_proj_2.view(batch_size,seq_len,self.num_heads,-1)
        v_proj_2 = v_proj_2.view(batch_size,seq_len,self.num_heads,-1)
        # this is important, apply on RoPE part too
        k_rope_2 = k_rope_2.view(batch_size, seq_len, self.num_heads, -1)
        q_rope_2 = q_rope_2.view(batch_size, seq_len, self.num_heads, -1)


        # apply rotary position embeddings
        rotary_emb =  self.rotary_emb(seq_len,x.device)
        k_rope_2 = self.rotary_emb.apply_rotary_emb(k_rope_2, rotary_emb)
        q_rope_2 = self.rotary_emb.apply_rotary_emb(q_rope_2, rotary_emb)

        # Concatenate context and rope part of k and q projections
        k = torch.cat([k_proj_2 ,k_rope_2 ], dim=-1)
        q = torch.cat([q_proj_2 ,q_rope_2 ], dim=-1)

        # swap seq_len and num_heads for attention computation
        # [batch_size,seq_len,num_heads,head_dim] --> [batch_size,num_heads,seq_len,head_dim]
        k = k.transpose(1,2)
        q = q.transpose(1,2)
        v_proj_2  = v_proj_2 .transpose(1,2)

        # Apply Attention
        attn_output = F.scaled_dot_product_attention(q,k,v_proj_2,attn_mask=attention_mask,dropout_p=0,is_causal=True)

        # reshape and project output
        attn_output = attn_output.transpose(1,2).contiguous().view(batch_size,seq_len,self.hidden_size)

        return self.o_proj(attn_output)



### DeepSeekDecoder Layer

`DeepSeekDecoderLayer` implementation, here are the key aspects:

**Standard Transformer Decoder Structure**: It follows the classic pattern with two main sublayers, each with residual connections and layer normalization.

**Two Sublayers**:
1. **Multi-Head Latent Attention** (MHLA): Uses the compressed attention mechanism you implemented above
2. **Mixture of Experts (MoE)**: A DeepSeekMoE module that replaces the standard feedforward network

**Pre-Norm Architecture**: Layer normalization is applied *before* each sublayer (input_layernorm before attention, post_attention_layernorm before MoE), which is the modern approach for training stability.

**Residual Connections**: Both sublayers use residual connections (x = residual + sublayer(norm(residual))), allowing gradients to flow directly through the network.

**MoE Parameters**: The layer can be configured with multiple experts (num_experts), shared experts (num_shared_experts), and top-k routing to select which experts process each token.

In [3]:
class DeepSeekDecoderLayer(nn.Module):
    def __init__(self, hidden_size,num_heads,intermediate_size,compression_ratio,num_experts,num_shared_experts,top_k):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.intermediate_size = intermediate_size
        self.compression_ratio = compression_ratio
        self.num_experts = num_experts
        self.num_shared_experts = num_shared_experts
        self.top_k = top_k
        self.self_attn = MultiHeadLatentAttention(hidden_size,num_heads,compression_ratio)
        self.input_layernorm = DeepSeekRMSNorm(hidden_size)
        self.post_attention_layernorm = DeepSeekRMSNorm(hidden_size )
        self.mlp = DeepSeekMoE(hidden_size, intermediate_size, num_experts,num_shared_experts, top_k)

    def forward(self,x,attention_mask=None):
        residual = x
        x = residual + self.self_attn(self.input_layernorm(x),attention_mask)
        residual = x
        x = residual + self.mlp(self.post_attention_layernorm(x))

        return x

### DeepSeek Expert

`DeepSeekExpert` implementation, here are the key aspects:

**Gated FFN Architecture**: This uses a gated feedforward network (also called SwiGLU), which is more expressive than a standard two-layer FFN.

**Three Linear Projections**:
- **gate_proj**: Projects from `hidden_size` to `intermediate_size` and applies activation
- **up_proj**: Projects from `hidden_size` to `intermediate_size` (parallel path)
- **down_proj**: Projects back from `intermediate_size` to `hidden_size`

**Element-wise Gating**: The output combines two paths multiplicatively: `down_proj(act_fn(gate_proj(x)) * up_proj(x))`. The gate path controls which information from the up path flows through.

**SiLU Activation**: Uses Sigmoid Linear Unit (also called Swish), which is `x * sigmoid(x)`, providing smooth, non-monotonic activation that works well in deep networks.

This expert design is what gets replicated multiple times in the MoE layer.

In [4]:
class DeepSeekExpert(nn.Module):
    def __init__(self,hidden_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size,intermediate_size,bias=False)
        self.down_proj = nn.Linear(intermediate_size,hidden_size,bias=False)
        self.up_proj = nn.Linear(hidden_size,intermediate_size,bias=False)
        self.act_fn = nn.SiLU()

    def forward(self,x):
        x_gate = self.act_fn(self.gate_proj(x))
        x_up = self.up_proj(x)
        x = self.down_proj(x_gate * x_up)

        return x


### Rotary Embedding

Here are the key aspects of `LlamaRotaryEmbedding`:

**Purpose**: Encodes positional information into query and key vectors by rotating them in a way that captures relative positions between tokens.

**Frequency Computation**: Creates inverse frequencies using `1/(10000^(i/dim))` for different dimension pairs, where lower dimensions get higher frequencies and higher dimensions get lower frequencies.

**Two-Stage Process**:
1. **forward()**: Generates cos/sin values for each position in the sequence by computing outer product of position indices with inverse frequencies
2. **apply_rotary_emb()**: Applies the actual rotation to input tensors

**Rotation Mechanism**: Splits the input into even/odd dimension pairs, then applies 2D rotation formula:
- `rotated_even = even * cos - odd * sin`
- `rotated_odd = even * sin + odd * cos`

**Key Advantage**: Unlike absolute position embeddings, RoPE naturally encodes relative positions through the rotation angles, which helps the model understand token relationships regardless of absolute position.

In [5]:
class LlamaRotaryEmbedding(nn.Module):
    def __init__(self,dim,max_position_embeddings=2048):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        indices = torch.arange(0,self.dim,2)
        inv_freq = 1/(10000.0**((indices)/self.dim))
        self.register_buffer("inv_freq",inv_freq)

    def forward(self,seq_len,device="cpu"):

        pos_indices = torch.arange(seq_len,device=device)
        angles = torch.outer(pos_indices.float(), self.inv_freq)
        return torch.cos(angles), torch.sin(angles)


    def apply_rotary_emb(self, x, rotary_emb):
        cos, sin = rotary_emb
        # Reshape cos/sin for broadcasting
        cos = cos.unsqueeze(0).unsqueeze(2)  # [1, seq_len, 1, dim//2]
        sin = sin.unsqueeze(0).unsqueeze(2)  # [1, seq_len, 1, dim//2]

        # Step 1: Split x into two halves (even and odd dimensions)
        # x shape: [..., dim]
        # x1: even indices [0, 2, 4, ...]
        # x2: odd indices [1, 3, 5, ...]
        x1 = x[...,0::2]
        x2 = x[...,1::2]

        # Step 2: Apply rotation formula
        # print(f"x1 shape: {x1.shape}")
        # print(f"cos shape: {cos.shape}")
        rotated_x1 = x1 * cos - x2 * sin
        rotated_x2 = x1 * sin + x2 * cos

        # Step 3: Interleave them back together
        # Stack and reshape to get original dimension ordering
        rotated_x = torch.stack([rotated_x1,rotated_x2],dim=-1).flatten(-2)

        return rotated_x

### RMS Norm

Here are the key aspects of `DeepSeekRMSNorm`:

**Purpose**: Normalizes activations to stabilize training, similar to LayerNorm but computationally simpler.

**RMS (Root Mean Square) Normalization**: Instead of normalizing by mean and standard deviation (like LayerNorm), it only uses the RMS of the values across the last dimension.

**Computation Steps**:
1. Calculate mean of squared values: `mean(x²)`
2. Take square root with epsilon for stability: `sqrt(mean(x²) + eps)`
3. Divide input by RMS and scale by learnable weight: `x/rms * weight`

**Learnable Scale Parameter**: The `weight` parameter allows the model to learn the optimal scale for each feature dimension after normalization.

**Advantages**: Faster than LayerNorm (no mean subtraction needed) while providing similar normalization benefits for training stability.

In [6]:
class DeepSeekRMSNorm(nn.Module):
    def __init__(self,hidden_size,eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self,x):
        rms = torch.sqrt(torch.mean(x**2,dim=-1, keepdim=True) + self.eps)
        x = x/rms*self.weight
        return x

### DeepSeek MoE

key aspects of `DeepSeekMoE`:

**Hybrid Expert Architecture**: Combines two types of experts:
- **Shared experts**: Always active for every token (process all inputs)
- **Routed experts**: Selectively activated based on router decisions

**Dynamic Routing**: A router network scores each token against all routed experts, then selects the top-k highest-scoring experts to process that token.

**Weighted Combination**: Each selected expert's output is weighted by its routing probability (via softmax), allowing the model to blend multiple expert opinions.

**Load Balancing Mechanisms**: Two approaches to prevent expert underutilization:
1. **Bias adjustment**: `update_bias_terms()` shifts routing biases to encourage underused experts
2. **Balance loss**: `compute_balance_loss()` adds a penalty when experts receive unequal token distributions

**Efficiency Trade-off**: While all shared experts run for every token, only top-k routed experts activate per token, reducing computation compared to running all experts while maintaining model capacity.

In [21]:
class DeepSeekMoE(nn.Module):
    def __init__(self,hidden_size, intermediate_size, num_experts, num_shared_experts, top_k):
        super().__init__()
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_experts = num_experts
        self.num_shared_experts = num_shared_experts
        self.top_k = top_k
        self.num_routed_experts = self.num_experts - self.num_shared_experts
        self.shared_experts = nn.ModuleList([DeepSeekExpert(self.hidden_size,self.intermediate_size) for _ in range(self.num_shared_experts)])
        self.routed_experts = nn.ModuleList([DeepSeekExpert(self.hidden_size,self.intermediate_size) for _ in range(self.num_routed_experts)])
        self.router = nn.Linear(self.hidden_size,self.num_routed_experts)
        self.routing_bias = nn.Parameter(torch.zeros(self.num_routed_experts))
        # init variable for tracking how much load goes to each expert
        self.expert_loads = torch.zeros(self.num_routed_experts)

    def forward(self,x):
        shared_output = torch.zeros_like(x)
        for expert in self.shared_experts:
            shared_output+= expert(x)

        routing_logits = self.router(x) + self.routing_bias
        routing_weights, top_k_indices = torch.topk(routing_logits,self.top_k,dim=-1)

        # store the routing probabilities and indices for complementry loss calculation
        self.routing_probs = F.softmax(routing_logits, dim=-1)
        self.top_k_indices_stored = top_k_indices

        # apply softmax to turn into probs
        routing_weights = F.softmax(routing_weights, dim=-1)

        # reset for each fwd pass
        self.expert_loads = torch.zeros(self.num_routed_experts, device=x.device)

        # Track expert selections (for debugging)
        self.expert_selections = torch.bincount(top_k_indices.flatten(), minlength=self.num_routed_experts)


        # now iterate through each column of experts in top k selection
        combined_op = torch.zeros_like(x)

        for k in range(self.top_k):
            expert_indices = top_k_indices[..., k]  # Which expert each token picked at position k
            scores = routing_weights[..., k]  # Weights for position k

            # now look at the experts selected
            for expert_id in range(self.num_routed_experts):
                mask = (expert_indices == expert_id)

                # Process tokens where mask is True
                if mask.any():
                    combined_op[mask]+= self.routed_experts[expert_id](x[mask])*scores[mask].unsqueeze(-1)
                    # update the load for expert (how many times it is being called)
                    self.expert_loads[expert_id]+=mask.sum()

        final_op = shared_output + combined_op

        return final_op

    def update_bias_terms(self, expert_loads,scaling_factor=0.1):
        # Calculate target load (equal distribution)
        target_load = 1.0/self.num_routed_experts

        # normalize expert loads
        expert_loads = expert_loads/expert_loads.sum(dim=-1)

        # Calculate load difference
        load_difference = expert_loads - target_load

        # update rate
        update_rate = scaling_factor*torch.abs(load_difference)

        # Update self.routing_bias.data
        self.routing_bias.data-=update_rate*load_difference

    def compute_balance_loss(self, alpha=0.01):
        """
        Compute sequence-wise balance loss to prevent extreme imbalance.
        Uses stored routing_probs and top_k_indices from forward pass.
        """
        routing_probs = self.routing_probs
        top_k_indices = self.top_k_indices_stored

        batch_size, seq_len, num_experts = routing_probs.shape
        balance_loss = 0.0

        for i in range(batch_size):
            expert_counts = torch.bincount(
                top_k_indices[i].flatten(),
                minlength=num_experts
            ).float()
            f_i = expert_counts / (self.top_k * seq_len)
            P_i = routing_probs[i].mean(dim=0)
            balance_loss += torch.sum(f_i * P_i)

        return alpha * balance_loss


### DeepSeek Model

key aspects of `DeepSeekModel`:

**Overall Architecture**: A decoder-only transformer language model that predicts the next token given input tokens.

**Four Main Components**:
1. **Embedding layer**: Converts token IDs to dense vector representations
2. **Stack of decoder layers**: Multiple `DeepSeekDecoderLayer` modules process the embeddings sequentially
3. **Final normalization**: RMSNorm applied after all decoder layers
4. **Language modeling head**: Projects hidden states back to vocabulary logits for next-token prediction

**Weight Tying**: The embedding layer and LM head share the same weight matrix (`lm_head.weight = embeddings.weight`), which reduces parameters and often improves performance.

**Key Innovations**: Combines two efficiency techniques throughout the decoder stack:
- Multi-Head Latent Attention (MHLA) for compressed attention computation
- Mixture of Experts (MoE) for sparse, scalable feedforward layers

**Configurable Depth**: The `num_layers` parameter controls how many decoder layers are stacked, with deeper models having more capacity but higher computational cost.

In [8]:
class DeepSeekModel(nn.Module):
    def __init__(self,vocab_size,hidden_size,num_layers,num_heads,intermediate_size,compression_ratio,num_experts,num_shared_experts,top_k):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size,hidden_size)
        self.decoder_layers = nn.ModuleList([DeepSeekDecoderLayer(hidden_size,num_heads,intermediate_size,compression_ratio,num_experts,num_shared_experts,top_k) for _ in range(num_layers)])
        self.norm = DeepSeekRMSNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size,vocab_size)
        # tie weights
        self.lm_head.weight = self.embeddings.weight

    def forward(self,input_ids,attention_mask=None):
        x = self.embeddings(input_ids)
        for decoder in self.decoder_layers:
            x = decoder(x,attention_mask)
        x = self.norm(x)
        x = self.lm_head(x)
        return x


### Loading SmolLM2

In [9]:
from transformers import AutoModelForCausalLM

hf_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")
print(hf_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

In [10]:
from transformers import AutoTokenizer
from datasets import load_dataset

# Load tokenizer for SmolLM2
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

### Copying SmolM2 params

In [11]:
vocab_size = hf_model.config.vocab_size
hidden_size = hf_model.config.hidden_size
num_layers = hf_model.config.num_hidden_layers
num_heads = hf_model.config.num_attention_heads
intermediate_size = hf_model.config.intermediate_size

In [22]:
model = DeepSeekModel(
    vocab_size=vocab_size,
    hidden_size=hidden_size,
    num_layers=6,
    num_heads=num_heads,
    intermediate_size=intermediate_size,
    compression_ratio=4,
    num_experts=8,
    num_shared_experts=1,
    top_k=2
)

### Copying embedding weights from SmolLM2

In [23]:
# Copy embedding weights
model.embeddings.weight.data.copy_(hf_model.model.embed_tokens.weight.data)

tensor([[-0.1177,  0.0278,  0.0481,  ..., -0.0228, -0.2520,  0.0283],
        [-0.0203, -0.0311, -0.0327,  ...,  0.0742, -0.0132, -0.1367],
        [-0.0212, -0.0337, -0.0278,  ...,  0.0669, -0.0122, -0.1367],
        ...,
        [ 0.1035, -0.0825, -0.0085,  ...,  0.0396,  0.0165, -0.1641],
        [ 0.0352, -0.0608,  0.0017,  ..., -0.1260, -0.0064, -0.0864],
        [ 0.0008,  0.0267,  0.1021,  ...,  0.1289, -0.1328, -0.0386]])

In [13]:
from datasets import load_dataset

# Load dataset in streaming mode
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

In [17]:
# Get a few samples
samples = list(dataset.take(3))
print(samples[0])

{'text': 'The Independent Jane\nFor all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.\nElizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.\nThe freedom she exhibited in finally accepting him in direct defiance of Lady Catherine and knowing her father would disapprove was unusual even for Austen. In her last book Anne Elliot is persuaded to refuse Captain Wentworth at Lady Russel’s insistence.\nAlthough Jane played by the rules of the day, all of her writing is infused with how she wanted life to be. She ‘screams’ her outrage at the limitations for women in Emma.\nWhen accosted by Mrs. Elton, Jane Fairfax says,\n“Excuse me, ma’am, but this is by no means my intention; I make no inquiry myself, and sho

In [18]:
from torch.utils.data import DataLoader

def tokenize_function(examples):
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length', return_tensors='pt')

tokenized_dataset = dataset.map(tokenize_function, batched=True)
train_dataloader = DataLoader(tokenized_dataset, batch_size=4)

In [24]:
# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")

Using device: cuda


In [25]:
def generate_text(prompt_text, max_length=50):
    model.eval()
    input_ids = tokenizer(prompt_text, return_tensors='pt')['input_ids'].to(device)

    with torch.no_grad():
        for _ in range(max_length):
            logits = model(input_ids)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

print(generate_text("Once upon a time"))

Once upon a time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time time


In [26]:
pip install pytorch-lightning tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 65.0 MB/s eta 0:00:00


In [35]:
import pytorch_lightning as pl
from torch.optim import AdamW

class DeepSeekLightningModule(pl.LightningModule):
    def __init__(self, model, learning_rate=3e-4):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        # Forward pass
        logits = self.model(input_ids)

        # Calculate loss
        loss = F.cross_entropy(
            logits[:, :-1, :].reshape(-1, logits.shape[-1]),
            input_ids[:, 1:].reshape(-1)
        )

        # After computing ce_loss
        balance_loss = 0
        for layer in self.model.decoder_layers:
            balance_loss += layer.mlp.compute_balance_loss(alpha=0.01)

        total_loss = loss + balance_loss

        # Log losses
        self.log('ce_loss', loss, prog_bar=True)
        self.log('balance_loss', balance_loss, prog_bar=True)
        self.log('train_loss', total_loss, prog_bar=True)

        # Update expert biases
        for layer in self.model.decoder_layers:
            layer.mlp.update_bias_terms(layer.mlp.expert_loads, scaling_factor=0.1)

        return total_loss

    # def configure_optimizers(self):
    #     optimizer = AdamW(self.parameters(), lr=self.learning_rate)
    #     return optimizer

    def configure_optimizers(self):
      optimizer = AdamW(self.parameters(), lr=self.learning_rate)

      scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
          optimizer,
          T_max=10000,  # Total training steps
          eta_min=1e-5
      )

      return {
          "optimizer": optimizer,
          "lr_scheduler": {
              "scheduler": scheduler,
              "interval": "step",
          }
      }

In [31]:
class FineWebEduDataModule(pl.LightningDataModule):
    def __init__(self, tokenizer, batch_size=8, num_workers=2):
        super().__init__()
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.num_workers = num_workers

    # Tokenize the dataset
    def tokenize_function(self, examples):
        self.tokenizer.pad_token = self.tokenizer.eos_token
        return self.tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length')

    def setup(self, stage=None):
        # Load and tokenize dataset
        # dataset = load_dataset("roneneldan/TinyStories", split="train[:5000]")
        self.train_dataset = dataset.map(self.tokenize_function, batched=True, remove_columns=['text'])
        self.train_dataset.set_format('torch') # donot return set as instance variable

    def collate_fn(self, batch):
      input_ids = torch.tensor([item['input_ids'] for item in batch])
      attention_mask = torch.tensor([item['attention_mask'] for item in batch])
      return {'input_ids': input_ids, 'attention_mask': attention_mask}

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                        num_workers=self.num_workers, collate_fn=self.collate_fn)




In [33]:
class FineWebEduDataModule(pl.LightningDataModule):
    def __init__(self, tokenizer, batch_size=8, num_workers=2):
        super().__init__()
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.num_workers = num_workers

    def tokenize_function(self, examples):
        self.tokenizer.pad_token = self.tokenizer.eos_token
        return self.tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length')

    def setup(self, stage=None):
        self.train_dataset = dataset.map(self.tokenize_function, batched=True)

    def collate_fn(self, batch):
        input_ids = torch.tensor([item['input_ids'] for item in batch])
        attention_mask = torch.tensor([item['attention_mask'] for item in batch])
        return {'input_ids': input_ids, 'attention_mask': attention_mask}

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                         num_workers=self.num_workers, collate_fn=self.collate_fn)


In [29]:
from pytorch_lightning.callbacks import ModelCheckpoint, RichProgressBar
from pytorch_lightning.loggers import TensorBoardLogger

# Create logger
logger = TensorBoardLogger("tb_logs", name="DeepSeek_Training")

# Create checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/",
    filename="deepseek-{step}",
    every_n_train_steps=5000,
    save_top_k=-1  # Save all checkpoints, or use save_top_k=1 with monitor='train_loss'
)

# Create trainer
trainer = pl.Trainer(
    max_steps=10000,
    logger=logger,
    callbacks=[checkpoint_callback, RichProgressBar()],
    precision="16-mixed",  # For autocast
    log_every_n_steps=100
)


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


In [36]:
# Initialize your module and data
lightning_model = DeepSeekLightningModule(model, learning_rate=3e-4)
data_module = FineWebEduDataModule(tokenizer, batch_size=8)

# Train!
trainer.fit(lightning_model, data_module)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type          ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ DeepSeekModel │  160 M │ eval │     0 │
└───┴───────┴───────────────┴────────┴──────┴───────┘

Trainable params: 160 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 160 M                                                                                                
Total estimated model params size (MB): 642                                                                        
Modules in train mode: 0                                                                                           
Modules in eval mode: 347                                                                                          
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=10000` reached.


In [37]:
def generate_text(model, tokenizer, prompt, max_length=50, temperature=0.8):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        for _ in range(max_length):
            logits = model(input_ids)
            next_token_logits = logits[0, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0])


In [41]:
prompt = "Once upon a time"
lightning_model.model.to('cuda')
trained_model = lightning_model.model
output = generate_text(trained_model, tokenizer, prompt)
print(output)

Once upon a time, one went way.
The first time a woman may have been suffering from the illness, but it was not through a community with a host of other conditions. However, the number of women who treated them will increase in pregnancy, and not 


### Calculating Perplexity

In [42]:
import math

ce_loss = 4.529 # CE loss after 10k steps
perplexity = math.exp(ce_loss)
print(f"Perplexity: {perplexity:.2f}")


Perplexity: 92.67


perplexity of 92.67 indicates the model is still somewhat uncertain - it's like choosing randomly from about 93 possible tokens on average.

What this means:

We've made progress from the starting perplexity of **~354 down to ~93**